# 🔗 Merge / Join con Pandas
### Python para Ciencia de Datos | UADE

---

Cuando los datos viven en tablas separadas, necesitamos **combinarlas** para analizarlas juntas.
En SQL esto se llama `JOIN`. En Pandas se hace con `pd.merge()`.

La idea central es unir dos DataFrames a través de una **columna clave** común.

```python
pd.merge(left, right, on="columna_clave", how="tipo_de_join")
```

| Tipo de join | Filas que incluye |
|---|---|
| `inner` | Solo las que tienen match en **ambas** tablas |
| `left`  | Todas las de la izquierda + las que hacen match a la derecha |
| `right` | Todas las de la derecha + las que hacen match a la izquierda |
| `outer` | Todas las de ambas tablas, con `NaN` donde no hay match |

---

> 💡 El parámetro `how` es el equivalente directo al tipo de JOIN en SQL.


### Diagrama de referencia

<img src="https://www.red-gate.com/simple-talk/wp-content/uploads/2025/09/Full-Outer-Join-featured-image-scaled.jpg" width="650">


In [ ]:
import pandas as pd

---
## 1. Los datasets de ejemplo

Vamos a trabajar con dos tablas típicas de un sistema de charts musicales:

- **`fact_chart`** — tabla de hechos: una fila por canción en un chart, con streams y posición
- **`dim_songs`** — dimensión: catálogo de canciones con nombre, artista y género

La columna `song_id` es la **clave** que conecta ambas tablas.

```
fact_chart          dim_songs
──────────          ──────────
song_id ──────────► song_id
chart_id            song_name
highest_position    artist
streams             genre
artist_followers    release_date
```


In [ ]:
# Tabla de hechos: canciones en charts
# song_id 1-9 → solo 5 están en dim_songs (1 a 5)
# song_id 6, 7, 8, 9 → NO están en dim_songs

fact_chart = pd.DataFrame({
    "song_id":                  [1,       2,       3,       4,       5,       6,       7,       8,       9],
    "chart_id":                 [101,     101,     102,     102,     103,     103,     104,     104,     105],
    "highest_charting_position":[1,       5,       10,      3,       8,       2,       7,       4,       6],
    "streams":                  [500000,  300000,  150000,  400000,  250000,  450000,  200000,  350000,  100000],
    "artist_followers":         [1000000, 800000,  500000,  1200000, 600000,  1100000, 700000,  900000,  400000],
})

print(f"fact_chart: {fact_chart.shape[0]} filas × {fact_chart.shape[1]} columnas")
fact_chart

In [ ]:
# Dimensión de canciones
# song_id 1-5 → están en fact_chart
# song_id 10, 11, 12 → NO están en fact_chart

dim_songs = pd.DataFrame({
    "song_id":   [1,          2,          3,          4,          5,          10,           11,           12],
    "song_name": ["Song A",   "Song B",   "Song C",   "Song D",   "Song E",   "Song X",     "Song Y",     "Song Z"],
    "artist":    ["Artist 1", "Artist 2", "Artist 3", "Artist 4", "Artist 5", "Artist X",   "Artist Y",   "Artist Z"],
    "genre":     ["Pop",      "Rock",     "Hip-Hop",  "Pop",      "Jazz",     "Electronic", "Pop",        "Rock"],
    "release_date": pd.to_datetime([
        "2020-01-01", "2020-02-01", "2020-03-01", "2020-04-01",
        "2020-05-01", "2021-01-01", "2021-02-01", "2021-03-01"
    ])
})

print(f"dim_songs:  {dim_songs.shape[0]} filas × {dim_songs.shape[1]} columnas")
dim_songs

### Solapamiento entre tablas

Es importante entender **cuántas claves están en ambas tablas** antes de hacer el merge:

| song_id | ¿Está en fact_chart? | ¿Está en dim_songs? |
|---|---|---|
| 1 – 5 | ✅ | ✅ |
| 6 – 9 | ✅ | ❌ |
| 10 – 12 | ❌ | ✅ |

Esto determina qué filas quedan en cada tipo de join.


---
## 2. INNER JOIN — solo los que hacen match

Conserva únicamente las filas donde `song_id` existe en **ambas** tablas.
Los `song_id` 6-9 (solo en fact_chart) y 10-12 (solo en dim_songs) se descartan.

```
fact_chart  ∩  dim_songs  →  song_id 1-5
```


In [ ]:
df_inner = pd.merge(fact_chart, dim_songs, on="song_id", how="inner")

print(f"fact_chart: {fact_chart.shape[0]} filas")
print(f"dim_songs:  {dim_songs.shape[0]} filas")
print(f"INNER JOIN: {df_inner.shape[0]} filas  ← solo los song_id presentes en ambas tablas")
df_inner.sort_values("song_id")

---
## 3. LEFT JOIN — todas las de la izquierda

Conserva **todas** las filas de `fact_chart` (izquierda).
Para los `song_id` 6-9 que no están en `dim_songs`, las columnas de esa tabla quedan como `NaN`.


In [ ]:
df_left = pd.merge(fact_chart, dim_songs, on="song_id", how="left")

print(f"fact_chart: {fact_chart.shape[0]} filas")
print(f"LEFT JOIN:  {df_left.shape[0]} filas  ← todas las de fact_chart")
df_left.sort_values("song_id")

In [ ]:
# Los NaN aparecen en las columnas de dim_songs para song_id 6-9
print("Filas con NaN en song_name (song_id sin match en dim_songs):")
df_left[df_left["song_name"].isna()]

---
## 4. RIGHT JOIN — todas las de la derecha

Conserva **todas** las filas de `dim_songs` (derecha).
Para los `song_id` 10-12 que no están en `fact_chart`, las columnas de esa tabla quedan como `NaN`.

> 💡 `right join` es equivalente a hacer `left join` con las tablas invertidas. En la práctica se usa poco — es más legible usar `left` e invertir el orden de los DataFrames.


In [ ]:
df_right = pd.merge(fact_chart, dim_songs, on="song_id", how="right")

print(f"dim_songs:   {dim_songs.shape[0]} filas")
print(f"RIGHT JOIN:  {df_right.shape[0]} filas  ← todas las de dim_songs")
df_right.sort_values("song_id")

---
## 5. FULL OUTER JOIN — todas las filas de ambas tablas

Conserva **todas** las filas de ambas tablas.
Donde no hay match, rellena con `NaN`.

```
fact_chart  ∪  dim_songs  →  song_id 1-12
```


In [ ]:
df_outer = pd.merge(fact_chart, dim_songs, on="song_id", how="outer")

print(f"fact_chart:       {fact_chart.shape[0]} filas")
print(f"dim_songs:        {dim_songs.shape[0]} filas")
print(f"FULL OUTER JOIN:  {df_outer.shape[0]} filas  ← unión completa")
df_outer.sort_values("song_id")

---
## 6. Indicador de origen — `indicator=True`

Con `indicator=True` Pandas agrega la columna `_merge` que indica de dónde vino cada fila:

| Valor | Significado |
|---|---|
| `left_only`  | Solo estaba en la tabla izquierda |
| `right_only` | Solo estaba en la tabla derecha |
| `both`       | Estaba en ambas tablas |

Es muy útil para **diagnosticar** el resultado del join antes de limpiarlo.


In [ ]:
df_outer_ind = pd.merge(fact_chart, dim_songs, on="song_id", how="outer", indicator=True)

df_outer_ind.sort_values("song_id")

In [ ]:
# Resumen por origen
df_outer_ind["_merge"].value_counts()

---
## 7. Comparativa de resultados

Un resumen del impacto de cada tipo de join sobre el número de filas:


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

resultados = {
    "INNER":  pd.merge(fact_chart, dim_songs, on="song_id", how="inner").shape[0],
    "LEFT":   pd.merge(fact_chart, dim_songs, on="song_id", how="left").shape[0],
    "RIGHT":  pd.merge(fact_chart, dim_songs, on="song_id", how="right").shape[0],
    "OUTER":  pd.merge(fact_chart, dim_songs, on="song_id", how="outer").shape[0],
}

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(resultados.keys(), resultados.values(),
              color=["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"], width=0.5)
ax.bar_label(bars, fmt="%d filas", padding=4, fontsize=11)
ax.set_ylabel("Cantidad de filas resultantes")
ax.set_title("Filas resultantes según tipo de join")
ax.set_ylim(0, max(resultados.values()) + 3)
plt.tight_layout()
plt.show()

print("Referencia:")
print(f"  fact_chart: {fact_chart.shape[0]} filas  (song_id 1–9)")
print(f"  dim_songs:  {dim_songs.shape[0]} filas  (song_id 1–5 y 10–12)")

---
## 🔑 Resumen

| Join | Filas resultantes | Cuándo usarlo |
|---|---|---|
| `inner` | Intersección | Solo querés filas con datos completos en ambas tablas |
| `left`  | Todas las de la izquierda | Querés conservar todos los registros base aunque no tengan match |
| `right` | Todas las de la derecha | Igual que left con tablas invertidas (poco usado) |
| `outer` | Unión completa | Querés ver todo, incluyendo los que no hacen match |

**Parámetros clave de `pd.merge()`:**

```python
pd.merge(
    left,           # DataFrame izquierdo
    right,          # DataFrame derecho
    on="col",       # columna clave común
    how="inner",    # tipo de join
    indicator=True  # agrega columna _merge para diagnóstico
)
```

> 🚀 **Próxima clase:** `pd.concat()` para apilar DataFrames verticalmente, y merge sobre múltiples claves.
